In [6]:
import os
from langchain.chat_models import init_chat_model
# groq_api_key=os.getenv("GROQ_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001EA20194BD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001EA20188350>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [7]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(...,description="the title of the movie")
    year:int=Field(...,description="this year the movie was released")
    director:str=Field(...,description="the director of the movie is")
    ratinng:float=Field(...,description="the movie rating out of 10 is")


In [8]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001EA20194BD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001EA20188350>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'the title of the movie', 'type': 'string'}, 'year': {'description': 'this year the movie was released', 'type': 'integer'}, 'director': {'description': 'the director of the movie is', 'type': 'string'}, 'ratinng': {'description': 'the movie rating 

In [4]:
model.invoke("provide details of the movie Inception")

AIMessage(content='<think>\nOkay, I need to provide details about the movie Inception. Let me start by recalling what I know. Inception is a 2010 science fiction action film directed by Christopher Nolan. The main actor is Leonardo DiCaprio, right? He plays Dom Cobb, a thief who steals information by entering people\'s dreams. The movie is about dream-sharing technology. \n\nThe concept of the film revolves around the ability to enter someone\'s subconscious through controlled dreaming. The team, led by Cobb, aims to plant an idea into a target\'s mind, which is called "inception." The antagonist might be Mal, Cobb\'s wife, who is a figment of his subconscious. There\'s also a character named Arthur, played by Joseph Gordon-Levitt, who is the team\'s enforcer. Ellen Page plays Ariadne, the architect who designs the dream layers. \n\nThe movie has a complex narrative structure, with multiple layers of dreams within dreams. Each layer has a different time dilation, so time moves slower a

In [5]:
response=model_with_structure.invoke("provide details of the movie inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', ratinng=8.8)

In [16]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    """a movie with details"""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie)
response=model_with_structure.invoke("provide details of movie inception")
response



Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [21]:
from pydantic import BaseModel , Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None,description="budget of the movie USD millions")

model_with_structure=model.with_structured_output(MovieDetails)

response=model_with_structure.invoke("Provide details of the movie Inception")

response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames')], genres=['Action', 'Science Fiction', 'Thriller'], budget=160.0)

In [23]:
from langchain_core.messages import Annotation
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """a movie with details"""
    title:Annotated[str,...,"the title of the movie "]
    year:Annotated[int,...,"the year in which movie was released"]
    director:Annotated[str,...,"the director of the movie "]
    rating: Annotated[float,...,"the movie's rating out of 10"]

model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("please provide the details of movie the avengers")

In [24]:
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [25]:
class Actor(TypedDict):
    name:str
    role:str
class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None,description="budget of the movie USD millions")

model_with_structure = model.with_structured_output(MovieDetails)

response= model_with_structure.invoke("provide details of the movie the avengers")

In [26]:
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}],
 'genres': ['Action', 'Science Fiction', 'Superhero'],
 'title': 'The Avengers',
 'year': 2012}

In [27]:
model.parse_raw

<bound method BaseModel.parse_raw of <class 'langchain_groq.chat_models.ChatGroq'>>

In [28]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

In [34]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information of a person"""
    name:str= Field(description="the name of the person")
    email:str=Field(description="the email address of the person is")
    phone:str=Field(description="the phone number of the person is")


model_with_structure= model.with_structured_output(ContactInfo)

result = model_with_structure.invoke("find contact info of Harsh Sahu")
result

BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': "I don't have access to personal contact information or private databases. If you need to contact Harsh Sahu, you might try professional networking platforms like LinkedIn or check if they have a public profile with contact details. Would you like guidance on how to craft a message if you find their contact information?"}}